# Cutoff method exploration

The cutoff is the threshold that separates the **linear segment** (ordinary days, light green to dark navy)
from the **log segment** (extraordinary spikes, deep blue to coral red). Both halves get equal visual weight,
so where you place the cutoff controls how much of the colour range is "spent" on spikes versus baseline variation.

This notebook compares different ways of computing the cutoff, using kayaking (strong clean seasonality)
and sourdough (flat baseline + one giant spike) as test cases — they stress different aspects of the problem.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from seasonal_spirals import fetch_pageviews, SeasonalSpiral
from seasonal_spirals._colormap import auto_cutoff

kayaking  = fetch_pageviews("Kayaking",  start="2018-01-01", end="2026-03-01")
sourdough = fetch_pageviews("Sourdough", start="2018-01-01", end="2026-03-01")

print(f"Kayaking  n={len(kayaking)}  min={kayaking.min():.0f}  median={kayaking.median():.0f}  p75={np.percentile(kayaking, 75):.0f}  p90={np.percentile(kayaking, 90):.0f}  max={kayaking.max():.0f}")
print(f"Sourdough n={len(sourdough)} min={sourdough.min():.0f}  median={sourdough.median():.0f}  p75={np.percentile(sourdough, 75):.0f}  p90={np.percentile(sourdough, 90):.0f}  max={sourdough.max():.0f}")

## Distribution overview

Histograms with candidate cutoff lines overlaid. Helps anchor intuition before looking at spirals.

In [ ]:
def candidate_cutoffs(values):
    """Return a dict of label -> cutoff value for several methods."""
    vals = np.asarray(values, dtype=float)
    vmin, vmax = vals.min(), vals.max()
    mean, std = vals.mean(), vals.std()
    return {
        "2x p75":        auto_cutoff(vals, vmin, vmax, cutoff_n=2.0, cutoff_percentile=75),
        "3x p75 (default)": auto_cutoff(vals, vmin, vmax, cutoff_n=3.0, cutoff_percentile=75),
        "4x p75":        auto_cutoff(vals, vmin, vmax, cutoff_n=4.0, cutoff_percentile=75),
        "3x p50":        auto_cutoff(vals, vmin, vmax, cutoff_n=3.0, cutoff_percentile=50),
        "3x p90":        auto_cutoff(vals, vmin, vmax, cutoff_n=3.0, cutoff_percentile=90),
        "p90 direct":    np.percentile(vals, 90),
        "p95 direct":    np.percentile(vals, 95),
        "mean+2std":     mean + 2 * std,
        "mean+3std":     mean + 3 * std,
    }

fig, axes = plt.subplots(1, 2, figsize=(16, 4))
colors = plt.cm.tab10(np.linspace(0, 0.9, 9))

for ax, (label, series) in zip(axes, [("Kayaking", kayaking), ("Sourdough", sourdough)]):
    vals = series.values
    ax.hist(vals, bins=60, color="steelblue", alpha=0.6, edgecolor="none")
    cuts = candidate_cutoffs(vals)
    for (name, val), col in zip(cuts.items(), colors):
        ax.axvline(val, color=col, linewidth=1.5, label=f"{name}: {val:.0f}")
    ax.set_title(label, fontsize=12)
    ax.set_xlabel("pageviews")
    ax.set_ylabel("days")
    ax.legend(fontsize=7.5, loc="upper right")

fig.suptitle("Distribution of daily pageviews with candidate cutoff lines", fontsize=13)
plt.tight_layout()
plt.show()

## Method 1: vary `cutoff_n` at fixed `cutoff_percentile=75`

`cutoff = cutoff_n * p75(data)`. Higher `n` pushes the threshold up, giving the linear segment more room.

In [ ]:
n_values = [1.5, 2.0, 3.0, 4.0, 5.0, 6.0]
datasets = [("Kayaking", kayaking), ("Sourdough", sourdough)]

fig, axes = plt.subplots(len(n_values), 2, figsize=(12, 6 * len(n_values)))

for row, n in enumerate(n_values):
    for col, (name, series) in enumerate(datasets):
        ax = axes[row, col]
        spiral = SeasonalSpiral(series, cutoff_n=n, cutoff_percentile=75)
        spiral.plot(ax=ax, show_year_labels=True, show_month_labels=True)
        cutval = spiral.cutoff
        ax.set_title(f"{name} | n={n}, p75 → cutoff={cutval:.0f}", fontsize=9, pad=8)

fig.suptitle("cutoff_n sweep (cutoff_percentile=75)", fontsize=13, y=1.002)
plt.tight_layout()
plt.show()

## Method 2: vary `cutoff_percentile` at fixed `cutoff_n=3`

`cutoff = 3 * p_m(data)`. A higher percentile anchors the threshold higher in the distribution.

In [ ]:
percentiles = [50, 60, 75, 85, 90, 95]

fig, axes = plt.subplots(len(percentiles), 2, figsize=(12, 6 * len(percentiles)))

for row, p in enumerate(percentiles):
    for col, (name, series) in enumerate(datasets):
        ax = axes[row, col]
        spiral = SeasonalSpiral(series, cutoff_n=3.0, cutoff_percentile=p)
        spiral.plot(ax=ax, show_year_labels=True, show_month_labels=True)
        cutval = spiral.cutoff
        ax.set_title(f"{name} | n=3, p{p} → cutoff={cutval:.0f}", fontsize=9, pad=8)

fig.suptitle("cutoff_percentile sweep (cutoff_n=3)", fontsize=13, y=1.002)
plt.tight_layout()
plt.show()

## Method 3: direct percentile as cutoff

Skip the multiplier entirely: use a high percentile directly as the cutoff.
The idea is that the top X% of days are "extraordinary" by definition.

In [ ]:
direct_percentiles = [80, 85, 90, 93, 95, 97]

fig, axes = plt.subplots(len(direct_percentiles), 2, figsize=(12, 6 * len(direct_percentiles)))

for row, p in enumerate(direct_percentiles):
    for col, (name, series) in enumerate(datasets):
        ax = axes[row, col]
        cutval = float(np.percentile(series.values, p))
        spiral = SeasonalSpiral(series, cutoff=cutval)
        spiral.plot(ax=ax, show_year_labels=True, show_month_labels=True)
        ax.set_title(f"{name} | p{p} direct → cutoff={cutval:.0f}", fontsize=9, pad=8)

fig.suptitle("Direct percentile cutoff (no multiplier)", fontsize=13, y=1.002)
plt.tight_layout()
plt.show()

## Method 4: mean + k * std

Classic outlier threshold. Works well when the distribution is roughly normal;
can be skewed by the spikes themselves.

In [ ]:
k_values = [1.0, 1.5, 2.0, 2.5, 3.0]

fig, axes = plt.subplots(len(k_values), 2, figsize=(12, 6 * len(k_values)))

for row, k in enumerate(k_values):
    for col, (name, series) in enumerate(datasets):
        ax = axes[row, col]
        vals = series.values.astype(float)
        cutval = float(np.clip(vals.mean() + k * vals.std(), vals.min() * 1.05, vals.max() * 0.95))
        spiral = SeasonalSpiral(series, cutoff=cutval)
        spiral.plot(ax=ax, show_year_labels=True, show_month_labels=True)
        ax.set_title(f"{name} | mean+{k}*std → cutoff={cutval:.0f}", fontsize=9, pad=8)

fig.suptitle("mean + k * std cutoff", fontsize=13, y=1.002)
plt.tight_layout()
plt.show()

## Summary table

Numeric cutoff values for each method across both datasets.

In [ ]:
rows = []
for name, series in datasets:
    vals = series.values.astype(float)
    vmin, vmax = vals.min(), vals.max()
    mean, std = vals.mean(), vals.std()
    p75 = np.percentile(vals, 75)
    row = {"dataset": name, "p75": p75, "mean": mean, "std": std}
    for n in [1.5, 2.0, 3.0, 4.0, 5.0]:
        row[f"n={n} p75"] = auto_cutoff(vals, vmin, vmax, cutoff_n=n, cutoff_percentile=75)
    for p in [75, 85, 90, 95]:
        row[f"p{p} direct"] = np.percentile(vals, p)
    for k in [1.5, 2.0, 3.0]:
        row[f"mean+{k}std"] = mean + k * std
    rows.append(row)

df = pd.DataFrame(rows).set_index("dataset").T
pd.options.display.float_format = "{:.0f}".format
df